In [1]:
import pandas as pd 
import numpy as np

In [22]:
# Included only 2007 and onward to unify scale as EF instead of F scale 
####  Future work: include type of tornado - frontal/ derecho 
#### which is counted if tornadoes span more than 1 county? 

#### BOONE county 2008 EF3 event 

In [4]:
df = pd.concat(map(pd.read_csv, ['raw/storm_data_search_results_2007-2014.csv','raw/storm_data_search_results_2015-2021.csv', 'raw/storm_data_search_results_2022-2024.csv']), ignore_index=True)
df['Date']= pd.to_datetime(df['BEGIN_DATE'])
df['County Name'] = df['CZ_NAME_STR'].str.split(' ').str[:-1].str.join(' ')
df = df.rename(columns={"CZ_FIPS": "County FIPS"})
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df = df.loc[df['Year']>=2007]
df = df.drop('Date',axis=1)
# df.to_csv('tornadoes_masterlist_2007Jan-2024Dec.csv')
# Check if the full list consisted all data from 2016 through 2025 - Adams county missing 2016 data 
# Not all counties are covered 
# Future work: add confidence in AQI levels by number of sites reporting - weighted by distance? 
df.keys()

Index(['EVENT_ID', 'CZ_NAME_STR', 'BEGIN_LOCATION', 'BEGIN_DATE', 'BEGIN_TIME',
       'EVENT_TYPE', 'MAGNITUDE', 'TOR_F_SCALE', 'DEATHS_DIRECT',
       'INJURIES_DIRECT', 'DAMAGE_PROPERTY_NUM', 'DAMAGE_CROPS_NUM',
       'STATE_ABBR', 'CZ_TIMEZONE', 'MAGNITUDE_TYPE', 'EPISODE_ID', 'CZ_TYPE',
       'County FIPS', 'WFO', 'INJURIES_INDIRECT', 'DEATHS_INDIRECT', 'SOURCE',
       'FLOOD_CAUSE', 'TOR_LENGTH', 'TOR_WIDTH', 'BEGIN_RANGE',
       'BEGIN_AZIMUTH', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'END_DATE',
       'END_TIME', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
       'EVENT_NARRATIVE', 'EPISODE_NARRATIVE', 'ABSOLUTE_ROWNUMBER',
       'County Name', 'Year', 'Month'],
      dtype='object')

In [5]:
df_all = df.loc[:,['County Name','County FIPS','Year','Month','EVENT_ID','TOR_F_SCALE', 'DEATHS_DIRECT', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_INDIRECT']]
len(df_all['County Name'].unique())

102

#### Monthly - each month clim

county name, month, all events, events EF2+, 

In [223]:
# df_idx = df_all.set_index(['County Name', 'Year', 'Month', 'TOR_F_SCALE'])
# valid_idx = df_all.groupby(['County Name','Year','Month'])['TOR_F_SCALE'].value_counts().index
# common_idx = valid_idx.intersection(df_idx.index)

In [222]:
### Per County Monthly 
# count_monthclim = df_all.groupby(['County Name','Month'])['TOR_F_SCALE'].value_counts().reset_index()
# dummies_monthclim = pd.get_dummies(count['TOR_F_SCALE']) #check dates 
# result_monthclim = dummies.mul(count['count'], axis=0)
# count_monthclim

In [6]:
count = df_all.groupby(['County Name','Year','Month'])['TOR_F_SCALE'].value_counts().reset_index()
dummies = pd.get_dummies(count['TOR_F_SCALE']) #check dates 
result = dummies.mul(count['count'], axis=0)

allevents = count.groupby(['County Name','Year','Month'])['count'].sum().reset_index().rename(columns={'count': 'All Events'})#.drop(count[(count['TOR_F_SCALE'].isin(['EFU','EF0','EF1']))].index)
ef234 = count.drop(count[(count['TOR_F_SCALE'].isin(['EFU','EF0','EF1']))].index).groupby(['County Name','Year','Month'])['count'].sum().reset_index().rename(columns={'count': 'Events of EF2+'})

monthly = pd.merge(allevents,ef234,on=['County Name','Year','Month'],how='outer').fillna(0)#.astype('Int64')#.astype(int)
deaths_injuries_mon = df_all.set_index(['County Name', 'Year', 'Month', 'TOR_F_SCALE']).loc[df_all.groupby(['County Name','Year','Month'])['TOR_F_SCALE'].value_counts().index][['DEATHS_DIRECT', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_INDIRECT']].groupby(['County Name','Year','Month']).sum().reset_index()

monthly = pd.merge(monthly,deaths_injuries_mon,how='outer')

In [7]:
monthly['Events of EF2+'] = monthly['Events of EF2+'].astype(int)
fip = df_all.groupby(['County Name'])['County FIPS'].unique().apply(lambda x: int(x[0]))

In [8]:
monthly = monthly.merge(fip,left_on='County Name', right_on='County Name', how='left')
new_order = ['County Name', 'Year', 'Month', 'County FIPS', 'All Events', 'Events of EF2+',
       'DEATHS_DIRECT', 'INJURIES_DIRECT', 'INJURIES_INDIRECT','DEATHS_INDIRECT']
monthlystats = monthly[new_order]
# monthlystats.to_csv('output/monthlystats_tornado_by_county_2007-2024.csv')

#### Yearly

In [11]:
yearly = monthly.groupby(['County Name','Year','County FIPS']).sum().drop('Month',axis=1).reset_index()
yearly

# yearly.to_csv('output/yearlystats_tornado_by_county_2007-2024.csv')

,County Name,Year,County FIPS,All Events,Events of EF2+,DEATHS_DIRECT,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_INDIRECT
0,ADAMS,2007,1,1,0,0,0,0,0
1,ADAMS,2008,1,3,0,0,0,0,0
2,ADAMS,2009,1,1,0,0,0,0,0
3,ADAMS,2023,1,1,0,0,0,0,0
4,ADAMS,2024,1,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
670,WOODFORD,2019,203,1,0,0,0,0,0
671,WOODFORD,2020,203,1,0,0,0,0,0
672,WOODFORD,2021,203,3,0,0,0,0,0
673,WOODFORD,2023,203,1,0,0,0,0,0


##### Summary

In [10]:
cols = ['All Events','Events of EF2+', 'DEATHS_DIRECT','INJURIES_DIRECT','INJURIES_INDIRECT','DEATHS_INDIRECT']
summary = yearly.groupby(['County Name','County FIPS']).sum(cols).drop('Year',axis=1).reset_index()
summary

# summary.to_csv('output/summarystats_tornado_by_county_2007-2024.csv')

In [11]:
summary['County Name'].values

array(['ADAMS', 'ALEXANDER', 'BOND', 'BOONE', 'BROWN', 'BUREAU',
       'CALHOUN', 'CARROLL', 'CASS', 'CHAMPAIGN', 'CHRISTIAN', 'CLARK',
       'CLAY', 'CLINTON', 'COLES', 'COOK', 'CRAWFORD', 'CUMBERLAND',
       'DE KALB', 'DE WITT', 'DOUGLAS', 'DU PAGE', 'EDGAR', 'EDWARDS',
       'EFFINGHAM', 'FAYETTE', 'FORD', 'FRANKLIN', 'FULTON', 'GALLATIN',
       'GREENE', 'GRUNDY', 'HAMILTON', 'HANCOCK', 'HARDIN', 'HENDERSON',
       'HENRY', 'IROQUOIS', 'JACKSON', 'JASPER', 'JEFFERSON', 'JERSEY',
       'JO DAVIESS', 'JOHNSON', 'KANE', 'KANKAKEE', 'KENDALL', 'KNOX',
       'LA SALLE', 'LAKE', 'LAWRENCE', 'LEE', 'LIVINGSTON', 'LOGAN',
       'MACON', 'MACOUPIN', 'MADISON', 'MARION', 'MARSHALL', 'MASON',
       'MASSAC', 'MCDONOUGH', 'MCHENRY', 'MCLEAN', 'MENARD', 'MERCER',
       'MONROE', 'MONTGOMERY', 'MORGAN', 'MOULTRIE', 'OGLE', 'PEORIA',
       'PERRY', 'PIATT', 'PIKE', 'POPE', 'PULASKI', 'PUTNAM', 'RANDOLPH',
       'RICHLAND', 'ROCK ISLAND', 'SALINE', 'SANGAMON', 'SCHUYLER',
       'SCO